# 00 · Data Ingestion (2015–2025 Raw Datasets)

**Project:** AgriShield — Early Warning System for Crop Yield & Livestock Forage Risk in Kenya  
**Notebook Goal:** Load and inspect our **10 raw datasets (2015–2025)** from `data_pipeline/data/` before cleaning them in the next notebook.

### Data Sources (Proposal Table 3.1)
1. **Geospatial & County Codes:** `kenya_counties.geojson` and `kenya_county_crosswalk.csv` (all 47 Kenyan counties).
2. **Forage & Vegetation (NASA MODIS):** `ken_ndvi_subnational_2015_2025_raw.csv` (pasture greenness / NDVI).
3. **Weather & Soil Moisture (CHIRPS & ERA5-Land):** Rainfall, temperature, and root-zone soil moisture (`2015–2025`).
4. **Crop Production & Demographics (KNBS & FAOSTAT):** Historical crop harvests (`2015–2025`) and county farming households (`2019 Census`).

---
### Step 1: Import Libraries & Check Our 10 Raw Files
In the cell below, we import `pandas`, `numpy`, `json`, and `Path`, point to our `../data` folder, and check that all 10 raw files are present.

In [16]:
import json
from pathlib import Path
import pandas as pd
import numpy as np

# Point to the folder where our 10 raw datasets are stored
# DATA_DIR = Path("../data")
DATA_DIR = Path("../data/raw")
# 00a writes to data/raw; legacy raw files are in data/. Check both.
DATA_DIRS = [RAW_DIR, DATA_DIR]
def find_raw(name):
    for d in DATA_DIRS:
        p = d / name
        if p.exists():
            return p
    return DATA_DIR / name  # fallback for error message


# List all files in the data folder and check their sizes
files = sorted(DATA_DIR.glob("*"))

file_summary = pd.DataFrame([
    {
        "File Name": f.name,
        "Size (KB)": round(f.stat().st_size / 1024, 1)
    }
    for f in files if not f.name.startswith(".")
])

print(f"Total raw files found: {len(file_summary)}")
display(file_summary)


Total raw files found: 9


,File Name,Size (KB)
0,MANIFEST.json,2.4
1,era5land_soil_moisture_runoff_2015_2025_raw.csv,335.0
2,faostat_kenya_crop_production_2015_2025_raw.csv,231.8
3,ken_chirps_rainfall_subnational_2015_2025_raw.csv,16322.0
4,ken_ndvi_subnational_2015_2025_raw.csv,4432.2
5,kenya_counties.geojson,6823.0
6,kenya_county_crosswalk.csv,1.4
7,modis_era5_climate_vegetation_monthly_2015_202...,275.4
8,nasa_power_monthly,1.1


### Step 2: Load Geospatial Boundaries & the 47-County Crosswalk

Why do we need these two files?
1. **`kenya_counties.geojson`**: Contains the map boundaries (polygons) for all **47 counties in Kenya**.
2. **`kenya_county_crosswalk.csv`**: Links different county IDs (`knbs_code`, `pcode` like `KE001`–`KE047`, and uppercase names) to the standard names of all **47 Kenyan counties** so we can merge all datasets accurately.

In [18]:
# 1. Load the 47-county crosswalk CSV

crosswalk_df = pd.read_csv(DATA_DIR / "kenya_county_crosswalk.csv")
print(f"Crosswalk shape: {crosswalk_df.shape} (Total counties: {crosswalk_df['county'].nunique()})")
display(crosswalk_df.head(10))

# 2. Load the Kenya Counties GeoJSON and verify all 47 counties are present

with open(DATA_DIR / "kenya_counties.geojson", "r") as f:
    kenya_geojson = json.load(f)

geojson_counties = sorted([feat["properties"]["county"] for feat in kenya_geojson["features"]])
print(f"\nTotal county polygons in GeoJSON: {len(geojson_counties)}")
print("First 10 counties in GeoJSON:", geojson_counties[:10])

Crosswalk shape: (47, 5) (Total counties: 47)


,county,knbs_code,pcode,desinventar_code,desinventar_name
0,Mombasa,1,KE001,28,MOMBASA
1,Kwale,2,KE002,19,KWALE
2,Kilifi,3,KE003,14,KILIFI
3,Tana River,4,KE004,40,TANA-RIVER
4,Lamu,5,KE005,21,LAMU
5,Taita-Taveta,6,KE006,39,TAITA-TAVETA
6,Garissa,7,KE007,6,GARISSA
7,Wajir,8,KE008,46,WAJIR
8,Mandera,9,KE009,24,MANDERA
9,Marsabit,10,KE010,25,MARSABIT



Total county polygons in GeoJSON: 47
First 10 counties in GeoJSON: ['Baringo', 'Bomet', 'Bungoma', 'Busia', 'Elgeyo-Marakwet', 'Embu', 'Garissa', 'Homa Bay', 'Isiolo', 'Kajiado']


### Step 3: Load Raw NASA MODIS NDVI & CHIRPS Rainfall Data (2015–2025)

Next, we load our two main satellite time-series datasets from `2015` to `2025`:
1. **`ken_ndvi_subnational_2015_2025_raw.csv`**: Raw 10-day (dekadal) **Normalized Difference Vegetation Index (NDVI)** from NASA MODIS (`vim` = vegetation greenness, `viq` = vegetation anomaly %). This measures pasture and crop health across all 47 counties.
2. **`ken_chirps_rainfall_subnational_2015_2025_raw.csv`**: Raw 10-day (dekadal) **CHIRPS v2.0 Rainfall** (`rfh` = 10-day rainfall in mm, `r1h` = 1-month rolling rainfall, `r3h` = 3-month rolling rainfall, `rfq` = rainfall anomaly %).

Notice that both files are raw and uncleaned: they contain a metadata tag row at the top, use `PCODE`s instead of county names, and include both county (`adm_level = 1`) and sub-county (`adm_level = 2`) records. We inspect these raw issues here so we know what to clean in `01_data_cleaning.ipynb`.

In [21]:
# 1. Load raw NASA MODIS NDVI (2015-2025

ndvi_raw = pd.read_csv(DATA_DIR / "ken_ndvi_subnational_2015_2025_raw.csv", low_memory=False)
print("====== Raw NASA MODIS NDVI (2015-2025) ======")
print(f"Shape: {ndvi_raw.shape}")
print(f"Columns: ", list(ndvi_raw.columns))
display(ndvi_raw.head(10))

# 2. Load raw CHIRPS Rainfall (2015-2025)

chirps_raw = pd.read_csv(DATA_DIR / "ken_chirps_rainfall_subnational_2015_2025_raw.csv", low_memory=False)
print("\n====== Raw CHIRPS Subnational Rainfall (2015-2025) ======")
print(f"Shape: {chirps_raw.shape}")
print(f"Columns: ", list(chirps_raw.columns))
display(chirps_raw.head(10))


====== Raw NASA MODIS NDVI (2015-2025) ======
Shape: (71118, 8)
Columns:  ['date', 'adm_level', 'adm_id', 'PCODE', 'n_pixels', 'vim', 'vim_avg', 'viq']


,date,adm_level,adm_id,PCODE,n_pixels,vim,vim_avg,viq
0,2002-07-01,1,51325,KE019,427.0,0.668536,0.684022,97.890240
1,2002-07-11,1,51325,KE019,427.0,0.658954,0.668657,98.649890
2,2002-07-21,1,51325,KE019,427.0,0.647139,0.657200,98.577350
3,2002-08-01,1,51325,KE019,427.0,0.636151,0.647714,98.342830
4,2002-08-11,1,51325,KE019,427.0,0.626647,0.640161,98.041870
5,2002-08-21,1,51325,KE019,427.0,0.616663,0.633099,97.593895
6,2002-09-01,1,51325,KE019,427.0,0.607007,0.626824,97.072180
7,2002-09-11,1,51325,KE019,427.0,0.598897,0.622517,96.487854
8,2002-09-21,1,51325,KE019,427.0,0.595050,0.621554,96.053410
9,2002-10-01,1,51325,KE019,427.0,0.599500,0.626105,96.064990



====== Raw CHIRPS Subnational Rainfall (2015-2025) ======
Shape: (133812, 15)
Columns:  ['date', 'adm_level', 'adm_id', 'PCODE', 'n_pixels', 'rfh', 'rfh_avg', 'r1h', 'r1h_avg', 'r3h', 'r3h_avg', 'rfq', 'r1q', 'r3q', 'version']


,date,adm_level,adm_id,PCODE,n_pixels,rfh,rfh_avg,r1h,r1h_avg,r3h,r3h_avg,rfq,r1q,r3q,version
0,1981-01-01,1,51325,KE019,427.0,7.372365,15.759407,NaN,NaN,NaN,NaN,59.598840,NaN,NaN,final
1,1981-01-11,1,51325,KE019,427.0,4.325527,19.294770,NaN,NaN,NaN,NaN,38.384920,NaN,NaN,final
2,1981-01-21,1,51325,KE019,427.0,5.569087,16.265417,17.266980,51.319595,NaN,NaN,49.700817,39.536823,NaN,final
3,1981-02-01,1,51325,KE019,427.0,5.882904,12.719282,15.777517,48.279470,NaN,NaN,61.418427,38.997230,NaN,final
4,1981-02-11,1,51325,KE019,427.0,17.180328,18.768618,28.632318,47.753320,NaN,NaN,93.317700,63.753933,NaN,final
5,1981-02-21,1,51325,KE019,427.0,3.142857,11.827791,26.206089,43.315693,NaN,NaN,48.389336,64.587900,NaN,final
6,1981-03-01,1,51325,KE019,427.0,14.159250,29.220140,34.482437,59.816550,NaN,NaN,55.988228,60.914127,NaN,final
7,1981-03-11,1,51325,KE019,427.0,66.765810,28.629663,84.067920,69.677600,NaN,NaN,213.400310,119.269930,NaN,final
8,1981-03-21,1,51325,KE019,427.0,141.128800,41.624825,222.053860,99.474630,265.52692,194.10991,313.414150,217.329200,135.86813,final
9,1981-04-01,1,51325,KE019,427.0,105.707260,73.952380,313.601870,144.206860,363.86182,252.30289,140.220290,213.530300,143.35706,final


### Step 4: Load Raw ERA5-Land Soil Moisture, Precipitation & MODIS Temperature Data (2015–2025)

In this step, we load the three meteorological and land-surface datasets that provide **temperature** and **root-zone soil moisture** across all **47 counties** from `2015` to `2025`:
1. **`era5land_soil_moisture_runoff_2015_2025_raw.csv`**: Raw Google Earth Engine (`ECMWF/ERA5-Land`) monthly county data containing `volumetric_soil_water_layer_1` (`0–7 cm` top-soil moisture) and `volumetric_soil_water_layer_2` (`7–28 cm` root-zone moisture).
2. **`era5land_precipitation_2015_2025_raw.csv`**: Raw `ERA5-Land` monthly precipitation (`mean`) across all 47 counties.
3. **`modis_era5_climate_vegetation_monthly_2015_2025_raw.csv`**: Raw monthly grid observations containing `max_temp_c`, `min_temp_c`, `mean_temp_c`, `humidity_rh_pct`, `soil_moisture_m3m3`, `rain_mm`, and `ndvi` (`2015–2025`).

**Raw Data Issues to Note for Cleaning Later:**
- The Earth Engine files store county names in uppercase under the column `FIRST_COUN` and include unnecessary GEE system columns (`system:index`, `COUNT`, `.geo`).
- The monthly grid file contains multiple `(lat, lon)` points per county and some border points labeled `"Unknown"`.

In [22]:
# 1. Load raw ERA5-Land Soil Moisture & Runoff (2015-2025)

era5_soil_raw = pd.read_csv(DATA_DIR / "era5land_soil_moisture_runoff_2015_2025_raw.csv", low_memory=False)
print("\n====== Raw ERA5-Lnd Soil Moisture & Runoff (2015-2025) ======")
print(f"Shape: {era5_soil_raw.shape} | Unique Counties: {era5_soil_raw['FIRST_COUN'].nunique()}")
display(era5_soil_raw.head(10))

# 2. Load raw ERA5-Land Monthly Precipitation (2015-2025)

era5_precip_raw = pd.read_csv(DATA_DIR / "era5land_precipitation_2015_2025_raw.csv", low_memory=False)
print("\n====== Raw ERA5-Lnd Monthly Precipitation (2015-2025) ======")
print(f"Shape: {era5_precip_raw.shape} | Unique Counties: {era5_precip_raw['FIRST_COUN'].nunique()}")
display(era5_precip_raw.head(10))

# 3. Load raw ERA5-Land Monthly Temperature (2015-2025)

climate_grid_raw = pd.read_csv(DATA_DIR / "modis_era5_climate_vegetation_monthly_2015_2025_raw.csv", low_memory=False)
print("\n====== Raw MODIS-ERA5 Climate & Vegetation (2015-2025) ======")
print(f"Shape: {climate_grid_raw.shape} | Data Range: {climate_grid_raw['month'].min()} to {climate_grid_raw['month'].max()}")
print("Columns: ", list(climate_grid_raw.columns))
display(climate_grid_raw.head(10))



====== Raw ERA5-Lnd Soil Moisture & Runoff (2015-2025) ======
Shape: (6721, 11) | Unique Counties: 47


,county,FIRST_COUN,year,month,gwetroot,gwetprof,qv2m,surface_runoff_mm_day,total_runoff_mm_day,volumetric_soil_water_layer_1,volumetric_soil_water_layer_2
0,Baringo,BARINGO,2015,1,0.42,0.44,7.74,NaN,NaN,0.42,0.44
1,Baringo,BARINGO,2015,2,0.40,0.41,8.92,NaN,NaN,0.40,0.41
2,Baringo,BARINGO,2015,3,0.39,0.40,9.23,NaN,NaN,0.39,0.40
3,Baringo,BARINGO,2015,4,0.53,0.52,13.03,NaN,NaN,0.53,0.52
4,Baringo,BARINGO,2015,5,0.65,0.66,13.35,NaN,NaN,0.65,0.66
5,Baringo,BARINGO,2015,6,0.65,0.68,12.95,NaN,NaN,0.65,0.68
6,Baringo,BARINGO,2015,7,0.61,0.64,12.25,NaN,NaN,0.61,0.64
7,Baringo,BARINGO,2015,8,0.59,0.62,11.98,NaN,NaN,0.59,0.62
8,Baringo,BARINGO,2015,9,0.52,0.54,11.88,NaN,NaN,0.52,0.54
9,Baringo,BARINGO,2015,10,0.51,0.52,12.55,NaN,NaN,0.51,0.52


FileNotFoundError: [Errno 2] No such file or directory: '../data/raw/era5land_precipitation_2015_2025_raw.csv'

### Step 5: Load Raw Crop Production (FAOSTAT & KNBS) & County Demographic Data (2015–2025)

Next, we load the three socioeconomic and agricultural production datasets from **Table 3.1** of our Project Proposal:
1. **`faostat_kenya_crop_production_2015_2025_raw.csv`**: Annual crop yield (`kg/ha`), production (`tonnes`), and area harvested (`ha`) from **FAOSTAT (`2015–2025`)**.
2. **`knbs_agricultural_production_2015_2025_raw.csv`**: Marketed and physical agricultural & livestock output from the **KNBS Statistical Abstract & Economic Survey Chapter 7 (`2015–2025*`)**.
3. **`knbs_2019_census_agriculture_households_raw.csv`**: County-level distribution of households practicing farming, crop production, livestock rearing, aquaculture, fishing, and irrigation from the **KNBS 2019 Census (Volume IV, Table 2.19)**.

**Raw Data Issues to Note for Cleaning Later:**
- The FAOSTAT file includes neighboring East African countries (`Uganda`, `Ethiopia`, `Tanzania`) alongside `Kenya`, plus flag columns (`A`, `E`).
- The KNBS production and census tables store numbers as comma-formatted text strings (e.g., `"8,506.0"`, `"12,043,016"`), have provisional column headers (`2024*`, `2025*`), and include a national `"KENYA"` total row (`County_Code = 000`) and slash/hyphen county names (`TAITA/TAVETA`, `ELGEYO/MARAKWET`).

In [ ]:
# 1. Load raw FAOSTAT Crop Production, Yield & Area Harvested (2015-2025)

faostat_raw = pd.read_csv(DATA_DIR / "faostat_kenya_crop_production_2015_2025_raw.csv")
print("=== 1. Raw FAOSTAT Crop Production & Yield (2015–2025) ===")
print(f"Shape: {faostat_raw.shape} | Countries in file: {faostat_raw['Country'].unique().tolist()}")
display(faostat_raw.head(5))

# 2. Load raw KNBS Chapter 7 Agricultural & Livestock Production (2015-2025*)

knbs_prod_raw = pd.read_csv(DATA_DIR / "knbs_agricultural_production_2015_2025_raw.csv")
print("\n=== 2. Raw KNBS Agricultural & Livestock Production (2015–2025*) ===")
print(f"Shape: {knbs_prod_raw.shape}")
display(knbs_prod_raw.head(6))

# 3. Load raw KNBS 2019 Census Agricultural & Livestock Households by County

census_demo_raw = pd.read_csv(DATA_DIR / "knbs_2019_census_agriculture_households_raw.csv")
print("\n=== 3. Raw KNBS 2019 Census Agricultural Households ===")
print(f"Shape: {census_demo_raw.shape} (Includes 1 national 'KENYA' total row + 47 counties)")
display(census_demo_raw.head(6))

### Step 6: Raw Data Quality & Audit Summary (What We Need to Clean Next)

Before moving to **`01_data_cleaning.ipynb`**, we generate a single **Raw Data Audit Summary Table** checking the rows, columns, missing values, and specific cleaning tasks required for each dataset.

In [ ]:
# Build a simple, clear audit summary of all tabular raw datasets
raw_datasets = {
    "County Crosswalk": (crosswalk_df, "Reference lookup for 47 counties (PCODE & KNBS codes)"),
    "MODIS NDVI (2015–2025)": (ndvi_raw, "Filter adm_level==1, map PCODE to county, aggregate dekads to monthly mean"),
    "CHIRPS Rainfall (2015–2025)": (chirps_raw, "Filter adm_level==1, map PCODE to county, sum dekadal rainfall per month"),
    "ERA5 Soil Moisture (2015–2025)": (era5_soil_raw, "Map uppercase FIRST_COUN to standard county, drop GEE system/.geo columns"),
    "ERA5 Precipitation (2015–2025)": (era5_precip_raw, "Map uppercase FIRST_COUN to standard county, drop GEE system/.geo columns"),
    "MODIS/ERA5 Climate Grid (2015–2025)": (climate_grid_raw, "Drop 'Unknown' border points, standardize county names, average per county-month"),
    "FAOSTAT Crops (2015–2025)": (faostat_raw, "Filter Country=='Kenya', keep clean numeric Yield & Production columns"),
    "KNBS Production (2015–2025*)": (knbs_prod_raw, "Strip commas from numbers, rename '2024*'/'2025*' to '2024'/'2025', reshape wide/long"),
    "KNBS 2019 Census Households": (census_demo_raw, "Drop 'KENYA' total row (000), strip commas, standardize 47 county names"),
}

audit_rows = []
for name, (df, cleaning_task) in raw_datasets.items():
    audit_rows.append({
        "Dataset": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Missing Values": int(df.isna().sum().sum()),
        "Duplicate Rows": int(df.duplicated().sum()),
        "Action Required in 01_data_cleaning.ipynb": cleaning_task,
    })

audit_df = pd.DataFrame(audit_rows)
display(audit_df)
print("\nNotebook 00 (Data Ingestion) Complete! Ready for 01_data_cleaning.ipynb.")